In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report
from xgboost import XGBClassifier

In [ ]:
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')
train_df = train_df.drop(columns='id', axis=1)

In [ ]:
# undersampling M class
m_class = train_df[train_df['spectral_type'] == 'M']
other_classes = train_df[train_df['spectral_type'] != 'M']
m_class_undersampled = m_class.sample(n=115000, random_state=42)
train_df = pd.concat([m_class_undersampled, other_classes], ignore_index=True)

In [ ]:
# oversampling O/B class
ob_class = train_df[train_df['spectral_type'] == 'O/B']
other_classes = train_df[train_df['spectral_type'] != 'O/B']
ob_class_oversampled = ob_class.sample(n=100000, replace=True, random_state=42)
train_df = pd.concat([ob_class_oversampled, other_classes], ignore_index=True)

In [ ]:
# remove outliers
columns = ['u', 'g', 'r', 'i', 'z', 'redshift']

for col in columns:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    train_df = train_df[(train_df[col] >= lower_bound) & (train_df[col] <= upper_bound)]

In [ ]:
# feature engineering
train_df['u_g'] = train_df['u'] - train_df['g']
train_df['g_r'] = train_df['g'] - train_df['r']
train_df['r_i'] = train_df['r'] - train_df['i']
train_df['i_z'] = train_df['i'] - train_df['z']
train_df['color_index'] = train_df['u'] - train_df['z']
train_df['redshift_color'] = train_df['redshift'] * train_df['color_index']
train_df['isNegativeAlpha'] = train_df['alpha'] < 0
train_df['isNegativeDelta'] = train_df['delta'] < 0

In [ ]:
# encode categorical
le_spectral = LabelEncoder()
le_galaxy = LabelEncoder()
train_df['spectral_type'] = le_spectral.fit_transform(train_df['spectral_type'])
train_df['galaxy_population'] = le_galaxy.fit_transform(train_df['galaxy_population'])

In [ ]:
# drop redundant features
train_df = train_df.drop(columns=['alpha', 'delta', 'r', 'g'])

In [ ]:
# normalize numeric features
categorical_cols = ['spectral_type', 'galaxy_population', 'class', 'isNegativeAlpha', 'isNegativeDelta']
numeric_cols = [col for col in train_df.columns if col not in categorical_cols]
scaler = StandardScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])

In [ ]:
# prepare X and y
y = train_df['class']
X = train_df.drop(columns=['class'])

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [ ]:
# train random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)
bal_acc_rf = balanced_accuracy_score(y_val, y_pred_rf)
print(f"Random Forest Balanced Accuracy: {bal_acc_rf:.4f}")

In [ ]:
# train xgboost
xgb = XGBClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1, tree_method='hist')
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_val)
bal_acc_xgb = balanced_accuracy_score(y_val, y_pred_xgb)
print(f"XGBoost Balanced Accuracy: {bal_acc_xgb:.4f}")

In [ ]:
print(classification_report(y_val, y_pred_rf, target_names=le_target.classes_))

In [ ]:
# prepare test data
test_ids = test_df['id'].copy()
test_df = test_df.drop(columns=['id'])

# same feature engineering
test_df['u_g'] = test_df['u'] - test_df['g']
test_df['g_r'] = test_df['g'] - test_df['r']
test_df['r_i'] = test_df['r'] - test_df['i']
test_df['i_z'] = test_df['i'] - test_df['z']
test_df['color_index'] = test_df['u'] - test_df['z']
test_df['redshift_color'] = test_df['redshift'] * test_df['color_index']
test_df['isNegativeAlpha'] = test_df['alpha'] < 0
test_df['isNegativeDelta'] = test_df['delta'] < 0

test_df['spectral_type'] = le_spectral.fit_transform(test_df['spectral_type'])
test_df['galaxy_population'] = le_galaxy.fit_transform(test_df['galaxy_population'])

test_df = test_df.drop(columns=['alpha', 'delta', 'r', 'g'])

numeric_cols_test = [col for col in test_df.columns if col not in ['spectral_type', 'galaxy_population', 'isNegativeAlpha', 'isNegativeDelta']]
test_df[numeric_cols_test] = scaler.transform(test_df[numeric_cols_test])

test_df = test_df[X.columns]

In [ ]:
# make predictions
predictions_encoded = rf.predict(test_df)
predictions = le_target.inverse_transform(predictions_encoded)

submission = pd.DataFrame({'id': test_ids, 'class': predictions})
submission.to_csv('submission.csv', index=False)
print("Submission file created!")